# Samuel Custom Voice Fine-Tune — Kaggle (GPU T4 x2, never TPU)

> **Accelerator:** `NvidiaTeslaT4` (2x T4 DDP)
> **Dataset:** `lydorandlydor/samuel-voice-samples`

**Repo is now PUBLIC** (`https://github.com/lydorianP/samuel-realtime-parrot`) — **no GH_TOKEN/HF_TOKEN secrets needed.**
> Persistence: **Save Output ON** (all files in `/kaggle/working` persisted)


In [ ]:
import pathlib
import os, sys, subprocess, time
print('='*60)
print('CELL 1: ENVIRONMENT & REPOSITORY SETUP (PUBLIC REPO, NO SECRETS)')
print('='*60)

print('[1/5] Installing uv...')
subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
print(subprocess.getoutput('uv --version'))

print('\n[2/5] Cloning public repository (no token needed, repo is now PUBLIC)...')
# Repo is now public: https://github.com/lydorianP/samuel-realtime-parrot
# No GH_TOKEN required, no SSH key needed
if not pathlib.Path("samuel-realtime-parrot").exists():
    res = subprocess.run(['git', 'clone', 'https://github.com/lydorianP/samuel-realtime-parrot.git'], capture_output=True, text=True)
    if res.returncode != 0:
        print('[FATAL] Git clone failed (public repo, no token needed)')
        print(res.stderr)
        sys.exit(1)
    print('✅ Repository cloned successfully (public).')
else:
    print('Repo already exists, pulling latest...')
    subprocess.run('cd samuel-realtime-parrot && git pull', shell=True)

os.chdir('samuel-realtime-parrot')
print(f'Working directory: {os.getcwd()}')

print('\n[3/5] Setting up Python 3.12...')
subprocess.run('uv python install 3.12', shell=True, check=True)

print('\n[4/5] Syncing dependencies...')
subprocess.run('uv sync', shell=True, check=True)

print('\n[5/5] Installing vendor/samuel training deps...')
print('Initializing git submodules...')
subprocess.run('git submodule update --init --recursive', shell=True, check=True)
print(subprocess.getoutput('ls -la vendor/samuel/ | head -n 20'))
print('Installing vendor/samuel with dependencies...')
# Fix torchvision mismatch: vendor pins torch 2.8, but Kaggle image has torch 2.10 + torchvision 0.24
# The error "torchvision::nms does not exist" is due to torchvision 0.24 expecting torch 2.10
# Fix: install compatible torchvision for torch 2.8 after vendor install, or pin correctly
res = subprocess.run('uv pip install -e vendor/samuel 2>&1 | tail -n 30', shell=True)
if res.returncode != 0:
    print('uv pip -e failed, trying pip install hydra-core explicitly...')
    subprocess.run('uv pip install hydra-core omegaconf wandb transformers librosa soundfile 2>&1 | tail -n 20', shell=True)
    subprocess.run('pip install -e vendor/samuel --no-build-isolation 2>&1 | tail -n 20', shell=True)
# Fix torchvision mismatch for torch 2.8
print('Fixing torchvision compatibility for torch 2.8...')
subprocess.run('pip install torchvision==0.21.0 --no-deps 2>&1 | tail -n 20', shell=True)
subprocess.run('uv pip install torchvision==0.21.0 --no-deps 2>&1 | tail -n 20', shell=True)
print('Verifying hydra and torch...')
print(subprocess.getoutput('uv run python -c \'import hydra; print(f"hydra {hydra.__version__}")\' 2>&1 | head -n 5'))
print(subprocess.getoutput('uv run python -c \'import torch; print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, Devices: {torch.cuda.device_count()}")\' 2>&1 | head -n 5'))
print(subprocess.getoutput('uv run python -c \'import torchvision; print(f"torchvision {torchvision.__version__}")\' 2>&1 | head -n 5'))
print('='*60)


In [ ]:
import pathlib, subprocess, numpy as np, json, os
print('='*60)
print('CELL 2: DATASET & PITCH CACHE PREPARATION')
print('='*60)

WAV_DIR = pathlib.Path('/kaggle/input/samuel-voice-samples')
if not WAV_DIR.exists():
    print(f'[WARN] Expected dataset at {WAV_DIR}, searching /kaggle/input...')
    for p in pathlib.Path('/kaggle/input').glob('*'):
        if list(p.rglob('*.wav')):
            WAV_DIR = p
            print(f'Found wavs at {WAV_DIR}')
            break

wavs = list(WAV_DIR.rglob('*.wav')) if WAV_DIR.exists() else []
print(f'Target WAV_DIR: {WAV_DIR}')
print(f'Found {len(wavs)} WAV files.')

if len(wavs) == 0:
    print('[FATAL] No WAV files found. Ensure \'lydorandlydor/samuel-voice-samples\' is attached in Settings.')
    sys.exit(1)

print('\n[1/2] Running prepare_custom_dataset.py...')
cmd = f'uv run python scripts/prepare_custom_dataset.py --wav-dir {WAV_DIR} --manifest manifests/custom.jsonl --pitch-cache manifests/pitch_cache/custom_spf512.npz --sample-rate 44100 --samples-per-frame 512'
print(f'Executing: {cmd}')
subprocess.run(cmd, shell=True, check=True)

print('\n[2/2] Verifying outputs...')
manifest_path = pathlib.Path('manifests/custom.jsonl')
print(f'Manifest lines: {sum(1 for _ in open(manifest_path))}')

cache_path = pathlib.Path('manifests/pitch_cache/custom_spf512.npz')
d = np.load(cache_path)
print(f'Pitch cache keys: {list(d.files)[:6]}...')
print(f'  n_files: {d["n_files"]}, sr: {d["sample_rate"]}, spf: {d["samples_per_frame"]}')
print('='*60)

In [ ]:
import subprocess, os
print('='*60)
print('CELL 3: DISTRIBUTED TRAINING (DDP on 2x T4)')
print('='*60)

print('[1/3] GPU Status:')
print(subprocess.getoutput('nvidia-smi'))

print('\n[2/3] Environment variables for DDP:')
os.environ['NCCL_DEBUG'] = 'INFO'
os.environ['PYTHONUNBUFFERED'] = '1'

print('\n[3/3] Launching torchrun...')
cmd = '''torchrun --standalone --nproc_per_node=2 -m samuel.train \
    run.name=kaggle_custom_voice_ft \
    data.manifest_path=manifests/custom.jsonl \
    data.pitch_cache_path=manifests/pitch_cache/custom_spf512.npz \
    batch_size=16 \
    optim.max_steps=5000 \
    optim.warmup_steps=500 \
    log.eval_every=500 \
    log.ckpt_every=1000 \
    log.wandb_mode=offline'''

print(f'Executing:\n{cmd}')
process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    print(f'\n[WARN] Training failed with exit code {process.returncode}. Attempting fallback to batch_size=8...')
    cmd_fallback = cmd.replace('batch_size=16', 'batch_size=8')
    process = subprocess.Popen(cmd_fallback, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    if process.returncode != 0:
        raise RuntimeError('Training failed even with batch_size=8')
print('='*60)

In [ ]:
import shutil, pathlib, subprocess
print('='*60)
print('CELL 4: EXTRACT & DOWNLOAD CHECKPOINT')
print('='*60)

ckpt_dir = pathlib.Path('runs/kaggle_custom_voice_ft')
candidates = sorted(ckpt_dir.parent.glob('kaggle_custom_voice_ft_*'))
if candidates:
    ckpt_dir = candidates[-1] / 'checkpoints'
else:
    ckpt_dir = pathlib.Path('runs/kaggle_custom_voice_ft/checkpoints')

print(f'Looking for checkpoints in: {ckpt_dir}')
if ckpt_dir.exists():
    for p in sorted(ckpt_dir.glob('*.pt')):
        print(f'  {p.name}: {p.stat().st_size/1024/1024:.1f} MB')
    
    latest = max(ckpt_dir.glob('*.pt'), key=lambda p: p.stat().st_mtime)
    dst = pathlib.Path('/kaggle/working/samuel_custom_last.pt')
    shutil.copy(latest, dst)
    
    src_cfg = latest.parent.parent / 'config.json'
    if src_cfg.exists():
        shutil.copy(src_cfg, '/kaggle/working/custom_config.json')
        print(f'Copied config.json to /kaggle/working/custom_config.json')
        
    print(f'\n✅ Checkpoint ready: {dst} ({dst.stat().st_size/1024/1024:.1f} MB)')
    subprocess.run(['cp', str(latest), 'samuel_custom_last.pt'])
    print('Copied to ./samuel_custom_last.pt (will persist if \'Save Output\' is ON)')
else:
    print('[FATAL] No checkpoint directory found.')
    subprocess.run('ls -R runs | head -n 100', shell=True)
print('='*60)

In [ ]:
import os, subprocess, pathlib
print('='*60)
print('CELL 5: FINALIZE (NO HF PUSH, PUBLIC REPO, PERSISTENCE)')
print('='*60)

print('Repo is now PUBLIC (https://github.com/lydorianP/samuel-realtime-parrot)')
print('No GH_TOKEN or HF_TOKEN secrets required for clone/push.')
print('Checkpoint will be in /kaggle/working/ and persisted via Save Output.')

# Verify output files exist in /kaggle/working (persistence)
import pathlib as _pathlib
ckpt = _pathlib.Path("/kaggle/working/samuel_custom_last.pt")
cfg = _pathlib.Path("/kaggle/working/custom_config.json")
if ckpt.exists():
    print(f"✅ Checkpoint in working: {ckpt} ({ckpt.stat().st_size/1024/1024:.1f} MB)")
else:
    # Try alternative location
    for p in _pathlib.Path("runs").rglob("*.pt"):
        print(f"Found checkpoint: {p} {p.stat().st_size/1024/1024:.1f} MB")
        # Copy to working for persistence
        import shutil
        shutil.copy(p, ckpt)
        print(f"Copied to {ckpt}")

if cfg.exists():
    print(f"✅ Config in working: {cfg}")
else:
    print("Config not yet in working, but should be after Cell 4")

print("\nPersistence: Ensure Kaggle UI has 'Save Output' ON (persistence to all).")
print("The checkpoint at /kaggle/working/samuel_custom_last.pt will be available via:")
print("  kaggle kernels output lydorandlydor/samuel-realtime-parrot-custom-train -p kaggle/output/")
print("And locally via: ./scripts/re_export_custom.sh kaggle/output/samuel_custom_last.pt")

# No HF push - repo is public, download via Kaggle output is sufficient
print("\nNo HF push (scrapped per Director). For manual HF push, run locally:")
print("  hf upload barbarabhb/samuel-realtime-parrot-custom kaggle/output/samuel_custom_last.pt --repo-type model")
print('='*60)
